In [44]:
pip install pandas

In [43]:
import pandas as pd, seaborn as sns
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

In [29]:
# 1. Fill missing age with median

df["Age"] = df["Age"].fillna(df["Age"].median())

In [30]:
# 2. Fill missing embarked with mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [35]:
# 3. Drop the 'Cabin' column (instead of 'deck')
# Using errors='ignore' prevents a KeyError if the cell is run multiple times
df = df.drop(columns=["Cabin"], errors="ignore")
print(df.shape)

(891, 11)


In [32]:
# 4. Remove duplicate rows
df = df.drop_duplicates()
print(df.shape)

(891, 11)


In [33]:
# 5. Convert Pclass to category type (before lowercase conversion)
df["Pclass"] = df["Pclass"].astype("category")
print(df["Pclass"].dtype)

category


In [34]:
# 6. Make all column names lowercase
df.columns = df.columns.str.lower()
print(df.columns.tolist())

['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'embarked']


In [36]:
# 7. Remove rows where fare is above 400 (basic outlier fix)
df = df[df["fare"] <= 400]
print(df.shape)

(888, 11)


In [37]:
# 8. Filter: only rows where age > 30
result = df[df["age"] > 30]
print(result.shape)

(302, 11)


In [38]:
# 9. Filter: only female passengers who survived (use query)
result = df.query("sex == 'female' and survived == 1")
print(result.shape)

(232, 11)


In [39]:
# 10. Select first 5 rows, first 3 columns using .iloc
print(df.iloc[0:5, 0:3])

   passengerid  survived pclass
0            1     False      3
1            2      True      1
2            3      True      3
3            4      True      1
4            5     False      3


In [40]:
# 11. Create a new column "family_size" = sibsp + parch
df["family_size"] = df["sibsp"] + df["parch"]
print(df[["sibsp","parch","family_size"]].head())

   sibsp  parch  family_size
0      1      0            1
1      1      0            1
2      0      0            0
3      1      0            1
4      0      0            0


In [41]:
# 12. Sort by fare, highest first
df_sorted = df.sort_values("fare", ascending=False)
print(df_sorted[["fare"]].head())

        fare
27   263.000
438  263.000
88   263.000
341  263.000
742  262.375


In [42]:
# 13. Count how many passengers per class
print(df["pclass"].value_counts())

pclass
3    491
1    213
2    184
Name: count, dtype: int64


In [47]:
# 13. Average age per passenger class
print(df.groupby("Pclass")["Age"].mean())

Pclass
1    38.233441
2    29.877630
3    25.140620
Name: Age, dtype: float64


In [51]:
# 14. Count of passengers per embarkation port
print(df.groupby("Embarked")["Survived"].count())

Embarked
C    168
Q     77
S    644
Name: Survived, dtype: int64


In [52]:
# 15. Survival rate (mean) per sex
print(df.groupby("Sex")["Survived"].mean())

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


In [53]:
# 16. Multiple aggregations at once: mean and max fare per class
print(df.groupby("Pclass")["Fare"].agg(["mean", "max"]))

             mean       max
Pclass                     
1       84.154687  512.3292
2       20.662183   73.5000
3       13.675550   69.5500


In [54]:
# 17. Different aggregation per column: mean age, sum of fare, per class
print(df.groupby("Pclass").agg({"Age": "mean", "Fare": "sum"}))

              Age        Fare
Pclass                       
1       38.233441  18177.4125
2       29.877630   3801.8417
3       25.140620   6714.6951


In [55]:
# 18. Create a small second dataset and merge (inner join)
class_info = pd.DataFrame({
    "Pclass": [1, 2, 3],
    "class_name": ["First", "Second", "Third"]
})

merged = df.merge(class_info, on="Pclass", how="inner")
print(merged[["Pclass", "class_name"]].head())

   Pclass class_name
0       3      Third
1       1      First
2       3      Third
3       1      First
4       3      Third


In [56]:
# 19. Left join — keep all original rows even if no match
class_info_partial = pd.DataFrame({
    "Pclass": [1, 2],
    "class_name": ["First", "Second"]
})

merged_left = df.merge(class_info_partial, on="Pclass", how="left")
print(merged_left["class_name"].isnull().sum())

491


In [57]:
# 20. Outer join — keep all rows from both sides
extra_info = pd.DataFrame({
    "Pclass": [1, 2, 4],
    "note": ["Top", "Mid", "Unknown"]
})

merged_outer = df.merge(extra_info, on="Pclass", how="outer")
print(merged_outer["note"].value_counts(dropna=False))

note
NaN        491
Top        216
Mid        184
Unknown      1
Name: count, dtype: int64


In [58]:
# 21. Average fare by class and sex (pivot_table)
pivot = df.pivot_table(values="Fare", index="Pclass", columns="Sex", aggfunc="mean")
print(pivot)

Sex         female       male
Pclass                       
1       106.125798  67.226127
2        21.970121  19.741782
3        16.118810  12.661633


In [59]:
# 22. Survival count by class and sex (pivot_table with sum)
pivot2 = df.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="sum")
print(pivot2)

Sex     female  male
Pclass              
1           91    45
2           70    17
3           72    47


In [60]:
# 23. Count of passengers by class and survival (crosstab)
print(pd.crosstab(df["Pclass"], df["Survived"]))

Survived    0    1
Pclass            
1          80  136
2          97   87
3         372  119


In [61]:
# 24. Crosstab with percentages instead of raw counts
print(pd.crosstab(df["Pclass"], df["Survived"], normalize="index"))

Survived         0         1
Pclass                      
1         0.370370  0.629630
2         0.527174  0.472826
3         0.757637  0.242363
